In [ ]:
# Install the Prophet forecasting library
# This only needs to run once per cluster session
%pip install prophet

In [ ]:
# Restart the Python kernel to activate the newly installed Prophet library
# Run this immediately after the pip install cell, before any other code
dbutils.library.restartPython()

In [ ]:
# Pull 4 years of daily GMV from the forecasting table at partner level
# ds = date column (required name for Prophet)
# y  = metric to forecast (required name for Prophet)
import pandas as pd

df = spark.sql("""
    SELECT
        DATE(loan_date_etz)     AS ds,
        partner_grouping_legacy AS partner,
        SUM(gmv_amount_usd)     AS y
    FROM commercial_analytics.public.forecasting
    WHERE
        loan_date_etz      >= '2022-01-01'
        AND loan_date_etz   <  CURRENT_DATE
        AND gmv_amount_usd IS NOT NULL
        AND gmv_amount_usd  >= 0
        AND partner_grouping_legacy IS NOT NULL
    GROUP BY 1, 2
    ORDER BY 1, 2
""").toPandas()

df["ds"] = pd.to_datetime(df["ds"])
print(f"Loaded {df['partner'].nunique()} partners, {len(df)} rows")
print(f"Date range: {df['ds'].min().date()} to {df['ds'].max().date()}")

In [ ]:
# Only forecast partners with >= 365 days of history
# and active within the last 60 days -- matching ai_forecast() logic
from datetime import datetime, timedelta

today      = pd.Timestamp.today().normalize()
cutoff_60  = today - pd.Timedelta(days=60)

partner_stats = (
    df.groupby("partner")
    .agg(
        days=("ds", "count"),
        last_date=("ds", "max")
    )
    .reset_index()
)

eligible = partner_stats[
    (partner_stats["days"] >= 365) &
    (partner_stats["last_date"] >= cutoff_60)
]["partner"].tolist()

ineligible = partner_stats[
    ~partner_stats["partner"].isin(eligible)
]["partner"].tolist()

print(f"Eligible for Prophet:  {len(eligible)} partners")
print(f"Ineligible (fallback): {len(ineligible)} partners")

In [ ]:
# Train a separate Prophet model per partner and forecast 365 days forward
from prophet import Prophet
import warnings
warnings.filterwarnings("ignore")

results = []
failed  = []

for partner in eligible:
    try:
        partner_df = df[df["partner"] == partner][["ds", "y"]].copy()

        m = Prophet(
    yearly_seasonality=False,
    weekly_seasonality=True,
    seasonality_mode="multiplicative",
    interval_width=0.80,
    changepoint_prior_scale=0.3
)
        m.add_seasonality(name='yearly', period=365.25, fourier_order=15)
        m.add_country_holidays(country_name="US")
        m.fit(partner_df)

        future   = m.make_future_dataframe(periods=365)
        forecast = m.predict(future)

        forecast["partner_grouping_legacy"] = partner
        results.append(
            forecast[["ds", "partner_grouping_legacy", "yhat", "yhat_lower", "yhat_upper"]]
        )

    except Exception as e:
        print(f"Failed: {partner} - {e}")
        failed.append(partner)

print(f"Completed: {len(results)} partners")
print(f"Failed:    {len(failed)} partners")
if failed:
    print(failed)

In [ ]:
# Partners with < 365 days history or inactive > 60 days
# get a weighted moving average fallback
fallback_results = []
forecast_dates = pd.date_range(
    start=today.replace(day=1),
    end=(today + pd.offsets.MonthEnd(1)).replace(day=1) + pd.offsets.MonthEnd(1),
    freq="D"
)

for partner in ineligible:
    partner_df = df[df["partner"] == partner][["ds", "y"]].copy()
    partner_df = partner_df.sort_values("ds")

    recent = partner_df[partner_df["ds"] >= today - pd.Timedelta(days=365)].copy()
    if len(recent) == 0:
        continue

    recent["rank"] = range(len(recent), 0, -1)
    wma = (recent["y"] * recent["rank"]).sum() / recent["rank"].sum()

    for fdate in forecast_dates:
        fallback_results.append({
            "ds":                       fdate,
            "partner_grouping_legacy":  partner,
            "yhat":                     max(round(wma, 2), 0),
            "yhat_lower":               max(round(wma * 0.85, 2), 0),
            "yhat_upper":               max(round(wma * 1.15, 2), 0),
        })

fallback_df = pd.DataFrame(fallback_results)
print(f"Fallback rows generated: {len(fallback_df)}")

In [ ]:
# Combine Prophet and fallback results
# Add forecast_horizon, run_date, and model columns
# Overwrite the forecast table so dashboard always has fresh data
current_month_start = today.replace(day=1)
next_month_start     = current_month_start + pd.DateOffset(months=1)
next_month_end       = next_month_start + pd.offsets.MonthEnd(1)

all_results = pd.concat(
    results + ([fallback_df] if len(fallback_df) > 0 else []),
    ignore_index=True
)

all_results["ds"] = pd.to_datetime(all_results["ds"])
all_results = all_results[
    (all_results["ds"] >= current_month_start) &
    (all_results["ds"] <= next_month_end)
].copy()

all_results["forecast_horizon"] = all_results["ds"].apply(
    lambda d: "current_month" if d < next_month_start else "next_month"
)
all_results["yhat"]       = all_results["yhat"].clip(lower=0).round(2)
all_results["yhat_lower"] = all_results["yhat_lower"].clip(lower=0).round(2)
all_results["yhat_upper"] = all_results["yhat_upper"].clip(lower=0).round(2)
all_results["run_date"]   = today
all_results["model"]      = all_results["partner_grouping_legacy"].apply(
    lambda p: "prophet" if p in eligible else "fallback_wma"
)

spark.createDataFrame(all_results).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("commercial_analytics.commercial_analytics.gmv_prophet_forecast")

print(f"Written {len(all_results)} rows for {all_results['partner_grouping_legacy'].nunique()} partners")

In [ ]:
# Quick sanity check on the forecast table
validation = spark.sql("""
    SELECT
        forecast_horizon,
        model,
        COUNT(DISTINCT partner_grouping_legacy) AS partners,
        COUNT(*)                                AS forecast_days,
        ROUND(SUM(yhat), 2)                     AS total_forecast_gmv,
        MIN(ds)                                 AS first_date,
        MAX(ds)                                 AS last_date
    FROM commercial_analytics.commercial_analytics.gmv_prophet_forecast
    GROUP BY forecast_horizon, model
    ORDER BY forecast_horizon, model
""")
display(validation)

In [ ]:
# Runs Prophet for each monthly cutoff from Jan 2025 to last month
# Each cutoff trains on data before that date and forecasts 365 days forward
# Results stored in gmv_prophet_backtest with training_cutoff column
# This gives true out-of-sample accuracy for the dashboard

import warnings
warnings.filterwarnings("ignore")

cutoff_dates = [pd.Timestamp("2026-01-01")]

tuned_partners = {
    "Carnival Cruise Line",
    "EFFY Jewelry",
    "ALG - B2C",
    "Virgin Voyages"
}

print(f"Running {len(cutoff_dates)} backtests across {len(eligible)} eligible partners")
print(f"Cutoff dates: {[c.date() for c in cutoff_dates]}")

backtest_results = []
backtest_failed  = []

for cutoff in cutoff_dates:
    for partner in eligible:
        try:
            partner_df = df[
                (df["partner"] == partner) &
                (df["ds"] < cutoff)
            ][["ds", "y"]].copy()

            if len(partner_df) < 180:
                continue

            if partner in tuned_partners:
                m = Prophet(
                    yearly_seasonality=False,
                    weekly_seasonality=True,
                    seasonality_mode="multiplicative",
                    interval_width=0.80,
                    changepoint_prior_scale=0.3
                )
                m.add_seasonality(name='yearly', period=365.25, fourier_order=15)
            else:
                m = Prophet(
                    yearly_seasonality=True,
                    weekly_seasonality=True,
                    seasonality_mode="multiplicative",
                    interval_width=0.80
                )

            m.add_country_holidays(country_name="US")
            m.fit(partner_df)

            future   = m.make_future_dataframe(periods=365)
            forecast = m.predict(future)

            oos = forecast[forecast["ds"] >= cutoff][
                ["ds", "yhat", "yhat_lower", "yhat_upper"]
            ].copy()
            oos["partner_grouping_legacy"] = partner
            oos["training_cutoff"]         = cutoff.date()
            oos["run_date"]                = today

            backtest_results.append(oos)

        except Exception as e:
            backtest_failed.append((cutoff.date(), partner, str(e)))

    print(f"Completed cutoff: {cutoff.date()}")

print(f"\nTotal result sets: {len(backtest_results)}")
print(f"Failed:            {len(backtest_failed)}")

In [ ]:
# Combine all backtest results and write to separate Delta table
# This table is  overwrite
backtest_df = pd.concat(backtest_results, ignore_index=True)
backtest_df["yhat"]       = backtest_df["yhat"].clip(lower=0).round(2)
backtest_df["yhat_lower"] = backtest_df["yhat_lower"].clip(lower=0).round(2)
backtest_df["yhat_upper"] = backtest_df["yhat_upper"].clip(lower=0).round(2)
backtest_df["training_cutoff"] = pd.to_datetime(backtest_df["training_cutoff"])
backtest_df["ds"]              = pd.to_datetime(backtest_df["ds"])

print(f"Total rows:    {len(backtest_df)}")
print(f"Partners:      {backtest_df['partner_grouping_legacy'].nunique()}")
print(f"Cutoffs:       {backtest_df['training_cutoff'].nunique()}")
print(f"Date range:    {backtest_df['ds'].min().date()} to {backtest_df['ds'].max().date()}")

spark.createDataFrame(backtest_df).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("commercial_analytics.commercial_analytics.gmv_prophet_backtest")

print("Written to gmv_prophet_backtest")

In [ ]:
# Adds a new cutoff each month so backtest history stays current
# Safe to run nightly -- skips if cutoff already exists

new_cutoff = today.replace(day=1) - pd.DateOffset(months=1)

existing = spark.sql("""
    SELECT DISTINCT training_cutoff
    FROM commercial_analytics.commercial_analytics.gmv_prophet_backtest
""").toPandas()["training_cutoff"].tolist()

existing = [pd.Timestamp(e) for e in existing]

if new_cutoff in existing:
    print(f"Cutoff {new_cutoff.date()} already exists - skipping")
else:
    print(f"Adding new cutoff: {new_cutoff.date()}")
    new_results = []

    for partner in eligible:
        try:
            partner_df = df[
                (df["partner"] == partner) &
                (df["ds"] < new_cutoff)
            ][["ds", "y"]].copy()

            if len(partner_df) < 180:
                continue

            m = Prophet(
                yearly_seasonality=False,
                weekly_seasonality=True,
                seasonality_mode="multiplicative",
                interval_width=0.80,
                changepoint_prior_scale=0.3
            )
            m.add_seasonality(name='yearly', period=365.25, fourier_order=15)
            m.add_country_holidays(country_name="US")
            m.fit(partner_df)

            future   = m.make_future_dataframe(periods=365)
            forecast = m.predict(future)

            oos = forecast[forecast["ds"] >= new_cutoff][
                ["ds", "yhat", "yhat_lower", "yhat_upper"]
            ].copy()
            oos["partner_grouping_legacy"] = partner
            oos["training_cutoff"]         = new_cutoff.date()
            oos["run_date"]                = today

            new_results.append(oos)

        except Exception as e:
            print(f"Failed: {partner} - {e}")

    if new_results:
        new_df = pd.concat(new_results, ignore_index=True)
        new_df["training_cutoff"] = pd.to_datetime(new_df["training_cutoff"])
        new_df["ds"]              = pd.to_datetime(new_df["ds"])

        spark.createDataFrame(new_df).write.format("delta").mode("append").saveAsTable("commercial_analytics.commercial_analytics.gmv_prophet_backtest")

        print(f"Appended {len(new_df)} rows for cutoff {new_cutoff.date()}")